# OpenPOM (MPNN-POM) on the Aromacle scaffold split

Run this notebook on a **Colab GPU runtime**. It does the "heavy lifting":

1. verify the GPU,
2. mount Google Drive (the dataset CSV and `split_seed42.json` live there),
3. install OpenPOM's dependencies (per the OpenPOM README),
4. train OpenPOM's MPNN-POM model on the **train** split, early-stop on the **val** split, and predict the **test** split,
5. copy `results/openpom_test_preds.csv` and `results/openpom_config.json` back to Drive.

All training logic lives in [`src/aromacle/gnn/train_openpom.py`](https://github.com/samragyi22/Aromacle/blob/main/src/aromacle/gnn/train_openpom.py),
which keeps every DGL / PyTorch / OpenPOM import isolated so the local classical-baseline pipeline still runs without them.

> Set the runtime to GPU first: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# Confirm a GPU is attached (expect a Tesla T4 table).
!nvidia-smi

In [ ]:
# Mount Google Drive (inputs are stored in MyDrive/Aromacle).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Confirm the dataset CSV and split JSON are present.
import os

DRIVE_DIR = '/content/drive/MyDrive/Aromacle'
print(sorted(os.listdir(DRIVE_DIR)))

## 1. Install dependencies

Versions follow the current OpenPOM README (Python 3.10, torch 2.4.0 / CUDA 12.4).
After this cell finishes, **restart the runtime** (Runtime > Restart session), then continue from the next cell.

In [ ]:
!pip install --pre deepchem
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
!pip install openpom scikit-learn pandas

In [ ]:
# Sanity check the install (run AFTER restarting the runtime).
import torch, dgl, deepchem, openpom
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available(), '| cuda', torch.version.cuda)
print('dgl', dgl.__version__, '| deepchem', deepchem.__version__)

## 2. Get the Aromacle code

Clone the repository and install it in editable mode. If the repository is private, either clone with a token
(`https://<TOKEN>@github.com/samragyi22/Aromacle.git`) or upload the `src/` folder and `pyproject.toml`.

In [ ]:
!git clone --depth 1 https://github.com/samragyi22/Aromacle.git /content/Aromacle || echo 'already cloned'
%cd /content/Aromacle
!pip install -e . -q

In [ ]:
import os

CSV_PATH = os.path.join(DRIVE_DIR, 'curated_GS_LF_merged_4983.csv')
SPLIT_PATH = os.path.join(DRIVE_DIR, 'split_seed42.json')
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(CSV_PATH), CSV_PATH
assert os.path.exists(SPLIT_PATH), SPLIT_PATH
print('csv  :', CSV_PATH)
print('split:', SPLIT_PATH)

## 3. Train MPNN-POM on the Aromacle split

Trains on the train indices, uses the val split for early stopping (patience on mean val ROC-AUC),
restores the best checkpoint, and writes test-split probabilities.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '-m', 'aromacle.gnn.train_openpom',
    '--csv', CSV_PATH,
    '--split', SPLIT_PATH,
    '--results-dir', RESULTS_DIR,
    '--epochs', '40',
    '--patience', '5',
    '--batch-size', '128',
    '--seed', '42',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
import pandas as pd

preds = pd.read_csv('results/openpom_test_preds.csv')
print('predictions:', preds.shape)  # (498, 138)
display(preds.head())

config = json.load(open('results/openpom_config.json'))
print('best epoch:', config['best_epoch'], '| val ROC-AUC:', round(config['best_val_roc_auc'], 4))
if config['history']:
    print('last epoch:', config['history'][-1])

In [ ]:
import shutil, os

for name in ('openpom_test_preds.csv', 'openpom_config.json'):
    shutil.copy(os.path.join(RESULTS_DIR, name), os.path.join(DRIVE_DIR, name))
print('copied results to', DRIVE_DIR)

## 4. Next: evaluate locally on identical terms

Download `openpom_test_preds.csv` from Drive into the repo's `results/`, then run:

```bash
python -m aromacle.evaluate_preds --preds results/openpom_test_preds.csv --model openpom --features mpnn
```

This uses the same evaluation logic as the classical baselines (same skipped labels, ROC-AUC, PR-AUC,
500-resample bootstrap CIs) and upserts the OpenPOM row into `results/baseline_metrics.csv`.

**Report honestly:** only claim OpenPOM "beats" a baseline if the bootstrap CIs support it.